In [1]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [2]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.sel(time=slice('1992-01-01', '2024-12-31')).to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [3]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [4]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [5]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [6]:
chirps_eastern_east_africa = chirps.sel(time=slice('1993-01-01', '2024-01-01'), latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
}

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

chirps_eastern_east_africa = chirps_eastern_east_africa.dropna(subset=['season'])

chirps_eastern_east_africa_monthly = chirps_eastern_east_africa.groupby(['year', 'month'])['precip'].mean().reset_index()

chirps_eastern_east_africa_season = chirps_eastern_east_africa.groupby(['year', 'season'])[['precip']].mean().reset_index()

In [7]:
def get_tercile_labels(chirps):
    tercile_list = chirps.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()
    bn_list = chirps.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    n_list = chirps.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    an_list = chirps.query(f'{tercile_list[1]} <= precip')['year'].to_list()
    season_dict = {'an': an_list, 'bn': bn_list, 'n': n_list}

    year_to_category_map = {}
    for category, year_list in season_dict.items():
        for year in year_list:
            year_to_category_map[year] = category

    chirps['tercile'] = chirps['year'].map(year_to_category_map)

    return chirps

In [8]:
labeled_chirps_seasonal = get_tercile_labels(chirps_eastern_east_africa_season)
labeled_chirps_monthly = chirps_eastern_east_africa_monthly.merge(labeled_chirps_seasonal[['year', 'tercile']], how='left', on='year')

In [9]:
nino_34 = sst_anomaly.query('-5 <= lat <= 5 and -170<= lon <= -120').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().rename({'normalized_sst_anomaly': 'nino_34'}, axis=1)

nino_4 = sst_anomaly.query('-5 <= lat <= 5 and lon <= -150 or -5 <= lat <= 5 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'nino_4'}, axis=1)

western_west_v = sst_anomaly.query('-15 <= lat <= 20 and 120 <= lon <= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'western_west_v'}, axis=1)

northern_west_v = sst_anomaly.query('20 <= lat <= 35 and lon <= -150 or 20 <= lat <= 35 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'northern_west_v'}, axis=1)

southern_west_v = sst_anomaly.query('-30 <= lat <= -15 and lon <= -150 or -30 <= lat <= -15 and lon >= 155').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'southern_west_v'}, axis=1)

SWIO = sst_anomaly.query('-50 <= lat <= -20 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'SWIO'}, axis=1)

IOD_west = sst_anomaly.query('-10 <= lat <= 10 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_west'}, axis=1)

IOD_east = sst_anomaly.query('-10 <= lat <= 0 and 90 <= lon <= 110').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_east'}, axis=1)

predictors = pd.concat([nino_34, nino_4, western_west_v, northern_west_v, southern_west_v, SWIO, IOD_west, IOD_east], axis=1)

In [36]:
predictors

,year,month,nino_34,nino_4,western_west_v,northern_west_v,southern_west_v,SWIO,IOD_west,IOD_east
0,1992,1,1.542857,0.397709,-1.497843,-0.821766,-1.280708,-0.330579,-0.727592,0.199368
1,1992,2,1.718218,0.811479,-1.160815,-1.085625,-0.752735,-0.499390,-0.949872,0.451130
2,1992,3,1.707231,0.689469,-1.536542,-0.925012,-1.181296,-0.627794,-1.010201,0.444106
3,1992,4,1.835929,0.590751,-1.209157,-1.364949,-1.528032,-0.504881,-0.972632,-0.236771
4,1992,5,1.721703,0.447074,-1.154650,-1.812978,-1.528706,0.159336,-0.415629,0.470575
...,...,...,...,...,...,...,...,...,...,...
391,2024,8,-0.076580,0.719398,1.221295,0.599335,0.475718,0.873648,1.439476,0.795291
392,2024,9,-0.247256,0.362220,1.161472,0.668998,0.650208,0.965824,1.504893,1.004850
393,2024,10,-0.210337,0.308758,1.280065,1.156696,0.409053,0.644471,1.040297,1.555399
394,2024,11,-0.170842,0.299586,1.449520,1.586774,0.302335,1.699368,0.937315,2.054945


In [39]:
def create_lead_month_mapping(target_season_name, target_start_month, lead_time):
    """
    Calculates the prediction month for a given target season and lead time.

    Args:
        target_season_name (str): The name of the target season (e.g., 'MAM').
        target_start_month (int): The numerical start month (1-12) of the target season.
        lead_time (int): The number of months lead time (e.g., 4).

    Returns:
        dict: A dictionary mapping the prediction month (int) to the target season name (str).
              Example: {11: 'MAM'} for a 4-month lead to March.
    """
    if not 1 <= target_start_month <= 12:
        raise ValueError("target_start_month must be between 1 and 12")
    if lead_time < 0:
        raise ValueError("lead_time cannot be negative")

    # Calculate the prediction month (1-12)
    # (target_start_month - lead_time - 1) gives the zero-based index offset
    # % 12 handles the wrap-around for negative results
    # + 1 converts back to 1-based month index
    prediction_month = (target_start_month - lead_time - 1) % 12 + 1

    return {prediction_month: target_season_name}

In [43]:
# --- Define the target season ---
season_name = 'MAM'
season_start = 3 # March is the 3rd month

season_info = {''}

# --- Generate dictionaries for leads 4 to 8 ---

# Lead time = 4 months (e.g., Nov -> Mar)
# P = (3 - 4 - 1) % 12 + 1 = (-2) % 12 + 1 = 10 + 1 = 11
four_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 4)

# Lead time = 5 months (e.g., Oct -> Mar)
# P = (3 - 5 - 1) % 12 + 1 = (-3) % 12 + 1 = 9 + 1 = 10
five_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 5)

# Lead time = 6 months (e.g., Sep -> Mar)
# P = (3 - 6 - 1) % 12 + 1 = (-4) % 12 + 1 = 8 + 1 = 9
six_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 6)

# Lead time = 7 months (e.g., Aug -> Mar)
# P = (3 - 7 - 1) % 12 + 1 = (-5) % 12 + 1 = 7 + 1 = 8
seven_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 7)

# Lead time = 8 months (e.g., Jul -> Mar)
# P = (3 - 8 - 1) % 12 + 1 = (-6) % 12 + 1 = 6 + 1 = 7
eight_month_lead_seasonal = create_lead_month_mapping(season_name, season_start, 8)

lead_maps = {}
for lead in range(4, 9): # Leads from 4 to 8 inclusive
    lead_maps[lead] = create_lead_month_mapping(season_name, season_start, lead)

In [58]:
# --- Refactored Code using a Loop ---

processed_dfs = {} # List to store the processed DataFrames for each lead time

# Loop through lead times from 4 to 8
for lead_time in range(4, 9):
    print(f"Processing lead time: {lead_time}")

    # Check if the mapping exists for the current lead time
    if lead_time not in lead_maps:
        print(f"Warning: Mapping for lead time {lead_time} not found. Skipping.")
        continue

    current_map = lead_maps[lead_time]

    # 1. Copy the base predictors DataFrame
    temp_df = predictors.copy()

    # 2. Map 'effect_season' using the correct lead time map
    temp_df['effect_season'] = temp_df['month'].map(current_map)

    year_adjustment = 0
    if lead_time >= season_start: # Check if lead time crosses start of year boundary
        year_adjustment = 1

    # Apply the adjustment to create the 'effect_year'
    # It's better to create 'effect_year' than to modify 'year' in place if 'year' is needed elsewhere
    if 'year' in temp_df.columns:
        temp_df['effect_year'] = temp_df['year'] + year_adjustment
    else:
        print(f"Warning: 'year' column not found for 'effect_year' calculation for lead {lead_time}.")

    # 3. Drop rows where mapping failed (NaN in 'effect_season') and drop 'month' column
    temp_df = temp_df.dropna(subset=['effect_season'])
    # Only drop 'month' if it exists, prevent errors
    if 'month' in temp_df.columns:
         temp_df = temp_df.drop('month', axis=1)
    else:
         print(f"Warning: 'month' column not found in temp_df for lead {lead_time} before dropping.")


    # --- Optional: Add lead time column if needed for identification later ---
    #temp_df['lead_time'] = lead_time

    # 4. Append the processed DataFrame to the list
    processed_dfs[f'lead_time_{lead_time}'] = temp_df

Processing lead time: 4
Processing lead time: 5
Processing lead time: 6
Processing lead time: 7
Processing lead time: 8


In [69]:
# Assume 'processed_dfs' is the list of DataFrames obtained from your previous loop.
# Each df in processed_dfs corresponds to lead times 4, 5, 6, 7, 8 respectively.
# Example: processed_dfs = [df_lead4, df_lead5, df_lead6, df_lead7, df_lead8]

# --- Step 1: Define Merge Keys and Lead Times ---

# !!! IMPORTANT: Verify these are the correct columns to uniquely identify rows
# !!!           These columns MUST exist in all DataFrames inside processed_dfs.
merge_keys = ['effect_season', 'effect_year'] # ADJUST AS NEEDED!


# List of lead times corresponding to the DataFrames in processed_dfs
lead_times = list(range(4, 9)) # Corresponds to leads 4, 5, 6, 7, 8

# Check if the number of dataframes matches the number of lead times
if len(processed_dfs) != len(lead_times):
    raise ValueError(f"Mismatch between number of dataframes ({len(processed_dfs)}) and lead times ({len(lead_times)})")

# --- Step 2: Rename columns in each DataFrame (excluding merge keys) ---

renamed_dfs = []
for key, df in processed_dfs.items():
    lead = key.split('_')[-1]
    suffix = f"_L{lead}"

    # Create a copy to avoid modifying the original dfs in the list if needed later
    df_renamed = df.copy().drop(['year'], axis=1)

    # Check if all merge keys exist in the current DataFrame
    missing_keys = [key for key in merge_keys if key not in df_renamed.columns]
    if missing_keys:
        raise ValueError(f"Merge key(s) {missing_keys} not found in DataFrame for lead {lead}")

    # Rename columns that are NOT in merge_keys
    cols_to_rename = {col: f"{col}{suffix}" for col in df_renamed.columns if col not in merge_keys}
    df_renamed = df_renamed.rename(columns=cols_to_rename)

    renamed_dfs.append(df_renamed)

# --- Step 3: Merge Horizontally ---

if not renamed_dfs:
    print("No dataframes to merge.")
    merged_predictors_seasonal = pd.DataFrame()
else:
    # Start with the first DataFrame
    merged_predictors_seasonal = renamed_dfs[0]

    # Iteratively merge the rest using an outer join
    for i in range(1, len(renamed_dfs)):
        try:
            merged_predictors_seasonal = pd.merge(
                merged_predictors_seasonal,
                renamed_dfs[i],
                on=merge_keys,
                how='outer' # Use 'outer' to keep all rows from all lead times
                            # Use 'inner' if you only want rows present in ALL lead times
            )
        except KeyError as e:
             print(f"\nError merging DataFrame for lead {lead_times[i]}. Missing key(s): {e}")
             print(f"Columns in left df: {merged_predictors_seasonal.columns.tolist()}")
             print(f"Columns in right df: {renamed_dfs[i].columns.tolist()}")
             # Handle error appropriately, e.g., break or continue

    # --- Alternative using reduce (more concise for many dataframes) ---
    # from functools import reduce
    # merge_func = lambda left, right: pd.merge(left, right, on=merge_keys, how='outer')
    # merged_predictors_seasonal = reduce(merge_func, renamed_dfs)

    print("\nHorizontal merge complete. Info of the merged DataFrame:")
    merged_predictors_seasonal.info()
    print("\nFirst 5 rows of merged DataFrame:")
    print(merged_predictors_seasonal.head())


Horizontal merge complete. Info of the merged DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nino_34_L4          33 non-null     float32
 1   nino_4_L4           33 non-null     float32
 2   western_west_v_L4   33 non-null     float32
 3   northern_west_v_L4  33 non-null     float32
 4   southern_west_v_L4  33 non-null     float32
 5   SWIO_L4             33 non-null     float32
 6   IOD_west_L4         33 non-null     float32
 7   IOD_east_L4         33 non-null     float32
 8   effect_season       33 non-null     object 
 9   effect_year         33 non-null     int32  
 10  nino_34_L5          33 non-null     float32
 11  nino_4_L5           33 non-null     float32
 12  western_west_v_L5   33 non-null     float32
 13  northern_west_v_L5  33 non-null     float32
 14  southern_west_v_L5  33 non-null     float32
 15  S

In [72]:
ml_data_seasonal = labeled_chirps_seasonal.merge(merged_predictors_seasonal, left_on=['year', 'season'], right_on=['effect_year', 'effect_season'], how='left').drop(['effect_year', 'effect_season'], axis=1)
ml_data_seasonal.to_csv('data/ml_data/ml_data_seasonal.csv')

,year,season,precip,tercile,nino_34_L4,nino_4_L4,western_west_v_L4,northern_west_v_L4,southern_west_v_L4,SWIO_L4,...,IOD_west_L7,IOD_east_L7,nino_34_L8,nino_4_L8,western_west_v_L8,northern_west_v_L8,southern_west_v_L8,SWIO_L8,IOD_west_L8,IOD_east_L8
0,1993,MAM,63.102024,n,-0.189563,-0.037570,-2.023475,-1.521014,-1.054530,0.061921,...,-1.650677,0.411593,0.385961,0.468095,-1.476969,-1.347927,-0.825886,-1.093551,-0.376044,0.678641
1,1994,MAM,70.791168,n,0.047522,0.076118,-1.169115,-1.689930,-0.889388,0.015313,...,-1.184849,-0.863439,0.371180,0.038182,-1.235332,-0.650575,-1.409905,-0.230075,-0.551691,-1.439500
2,1995,MAM,75.360275,an,0.963306,0.636810,-1.380513,-0.567587,-1.151014,-1.267918,...,-0.691272,-2.904368,0.410427,0.820223,-0.931489,-0.850618,-2.017670,0.034089,-0.608578,-2.023705
3,1996,MAM,66.239967,n,-0.872308,-0.556079,0.274520,-0.226184,0.380841,-0.340705,...,-1.167282,-0.003383,-0.266878,0.090527,-0.106994,-0.639811,-0.290482,-0.150222,-0.701315,0.402371
4,1997,MAM,81.642693,an,-0.308767,-0.190948,-0.093523,-1.388809,-0.397407,0.009275,...,-1.715001,0.244491,-0.622393,-0.769205,0.482297,-0.101144,-0.507914,-0.718925,-1.610068,-0.222927
5,1998,MAM,73.752174,an,1.984847,0.519474,-1.195996,-0.696848,-1.436354,-0.982311,...,-0.088452,-0.859536,2.315199,0.659606,-1.036561,-0.983373,-1.564697,-0.174725,-0.238690,-1.308930
6,1999,MAM,49.764690,bn,-1.180667,-1.583792,0.264448,-0.192127,1.765000,0.031293,...,-0.224299,1.213070,-1.371518,-1.325699,0.743412,-1.110201,0.065739,-0.089107,-0.051326,1.375072
7,2000,MAM,48.286407,bn,-1.281818,-1.411659,-0.498502,-0.107322,-0.455050,-0.236200,...,-0.652483,-1.028229,-1.727616,-1.880570,-0.570223,-0.731995,0.166279,-0.346727,-0.476965,-0.883098
8,2001,MAM,55.360474,bn,-0.632900,-0.761387,0.186136,0.011090,0.146958,-0.609975,...,-0.247704,0.005240,-0.972854,-1.253484,-0.970369,-0.329930,-0.927557,-0.133258,-0.471827,-0.689402
9,2002,MAM,66.756470,n,-0.264416,0.162081,-0.176892,0.082894,0.353659,-0.339551,...,-0.577058,0.268345,-0.016356,0.140887,-0.114005,0.481321,-0.430759,0.056236,-0.877177,0.691247


In [31]:
four_month_lead_monthly = {9: 3, 10: 4, 11: 5}
#{10: 3, 11: 4, 12: 5}

seven_month_lead_monthly = {6: 3, 7: 4, 8: 5}
#{7: 3, 8: 4, 9: 5}

predictors_5_lead_monthly = predictors.copy()
predictors_8_lead_monthly = predictors.copy()
predictors_5_lead_monthly['effect_month'] = predictors_5_lead_monthly['month'].map(five_month_lead_monthly)
predictors_8_lead_monthly['effect_month'] = predictors_8_lead_monthly['month'].map(eight_month_lead_monthly)

predictors_with_lead_monthly = predictors_8_lead_monthly.dropna(subset=['effect_month']).merge(predictors_5_lead_monthly.dropna(subset=['effect_month']), on=['year', 'effect_month'], suffixes=('_8_lead', '_5_lead'))

predictors_with_lead_monthly['effect_year'] = predictors_with_lead_monthly['year'] + 1

ml_data_monthly = labeled_chirps_monthly.merge(predictors_with_lead_monthly.drop(['year'], axis=1), left_on=['year', 'month'], right_on=['effect_year', 'effect_month'], how='left').drop(['effect_year', 'effect_month'], axis=1)

In [32]:
ml_data_monthly

,year,month,precip,tercile,month_8_lead,nino_34_8_lead,nino_4_8_lead,western_west_v_8_lead,northern_west_v_8_lead,southern_west_v_8_lead,...,IOD_east_8_lead,month_5_lead,nino_34_5_lead,nino_4_5_lead,western_west_v_5_lead,northern_west_v_5_lead,southern_west_v_5_lead,SWIO_5_lead,IOD_west_5_lead,IOD_east_5_lead
0,1993,3,12.088976,n,5,1.721703,0.447074,-1.154650,-1.812978,-1.528706,...,0.470575,8,0.108961,0.207684,-0.764448,-1.047540,-0.595462,-0.185888,-1.650677,0.411593
1,1993,4,69.710571,n,6,1.012786,0.428057,-1.178276,-1.637817,-1.170805,...,1.060601,9,-0.045081,0.182124,-0.790908,-1.018964,-0.394103,-0.500353,-1.386017,0.109431
2,1993,5,107.506531,n,7,0.385961,0.468095,-1.476969,-1.347927,-0.825886,...,0.678641,10,-0.243317,0.095080,-1.099177,-1.332042,-0.876752,-0.157934,-0.864180,0.185914
3,1994,3,16.170179,n,5,1.302434,0.035594,-1.920396,-0.532841,-1.743736,...,-0.753161,8,0.185317,0.125443,-1.498355,-0.980822,-1.463491,0.029869,-1.184849,-0.863439
4,1994,4,108.501205,n,6,0.635170,-0.013306,-1.248337,-0.679677,-1.561254,...,-0.965634,9,0.396386,0.350320,-1.494324,-0.998205,-1.424005,-0.464487,-1.330735,-0.393922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,2022,4,71.189896,bn,6,-0.473417,-0.236356,0.190674,0.440463,0.480127,...,0.775254,9,-0.587275,-0.630948,0.975200,0.542745,0.713782,0.033757,0.259481,0.361383
89,2022,5,46.031528,bn,7,-0.565679,-0.329533,0.592368,0.213540,0.727772,...,0.692127,10,-0.830146,-0.745362,1.084044,0.609405,0.956132,0.515981,-0.120459,0.840800
90,2023,3,78.101036,an,5,-1.749099,-1.308692,0.907826,1.244493,1.186736,...,0.943842,8,-1.163965,-1.603123,1.125830,0.952384,1.720167,0.467901,-0.983610,1.340814
91,2023,4,148.308960,an,6,-1.291519,-1.271019,1.406188,1.372706,1.500004,...,1.322234,9,-1.140091,-1.595459,1.409911,0.654021,1.738623,0.368002,-0.933255,0.968814
